# KHUDA 10기 ML세션
Date : 2026.07.22 (수)

In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve

In [26]:
# dataset repository path
url = "https://raw.githubusercontent.com/chaegyeong/KHUDA_ML_10th/7de72b3492f4bb64923bf99c4fdb12b5e1bea6ee/KHUDA_ML10th_WEEK2/ML10th_WEEk2.csv"

In [27]:
df = pd.read_csv(url)
df.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [28]:
### 건들지마세요
n=4
answer = [0] * 5

## Task 1
피마 인디언 당뇨병 데이터셋에는 결측치(NaN)가 존재하지 않습니다. 다만, 수치적으로 0이 될 수 없는 생체 지표 열들에 데이터 부재 대신 0으로 채워져 있음을 확인했습니다.

해당 df의 feature들은 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI' 입니다.

1. 대상 feature에서 값이 0인 데이터를 **제외한 해당 열의 중앙값으로** 대체합니다.

      *1-1 Hint  : 0인 값을 제외하고 중앙값을 구하거나, 0인 값들을 replace를 활용해 NaN(np.nan)으로 대체 후 처리하는 방식을 사용할 수 있습니다.*

2. 대체 후, BMI열의 전체 평균값을 구하여 소수점 둘째 자리까지 반올림(셋째 자리에서 반올림)하세요.
그리고, 해당 값을 answer[1]에 저장하세요.

In [29]:
## Input Box

features=[ 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI' ]

df[features]=df[features].replace(0,np.nan)
df[features] = df[features].fillna(df[features].median())
df_bmi_mean = round(df["BMI"].mean(),2)


In [30]:
answer[1] = df_bmi_mean
print(answer[1])

32.46


In [31]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0
764,2,122.0,70.0,27.0,125.0,36.8,0.340,27,0
765,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0
766,1,126.0,60.0,29.0,125.0,30.1,0.349,47,1


## Task 2

feature 'Age'가 30 이상이면 True, 미만이면 False를 갖는 파생 변수 'AgeOver30'을 생성하세요. 그 후 'AgeOver30'이 1이면서 동시에 'Outcome'이 1인, **즉, 30세 이상이면서 당뇨병 확진인** 환자의 수를 구하여 answer[2]에 저장하세요.

In [32]:
## Input Box
ans = 0

df['AgeOver30'] = (df['Age'] >= 30)

ans = len(df[(df['AgeOver30'] ==1) & (df['Outcome'] == 1)])

In [33]:
answer[2] = ans
print(answer[2])

184


## Task 3


Task 1에서 0을 중앙값으로 전처리한 데이터셋을 활용하여 로지스틱 회귀 모델을 학습하고 평가하세요.

1. 피처 선택: Outcome와 파생변수 AgeOver30을 제외한 나머지 8개 원본 피처를 독립변수($X$)로, Outcome을 종속변수($y$)로 설정하세요.

2. 스케일링: $X$ 데이터 전체에 StandardScaler를 적용하세요.

3. 데이터 분할: train_test_split을 이용해 학습/테스트 데이터를 나누세요.


    조건: test_size=0.2, random_state=42, stratify=y

4. 모델 학습: LogisticRegression(solver='liblinear') 모델을 생성하고 학습시키세요.

5. 평가 및 저장: 테스트 데이터셋에 대한 ROC-AUC 점수(양성 클래스 1일 확률 기반)를 구하고, 소수점 둘째 자리까지 반올림(셋째 자리에서 반올림)하여 answer[3]에 저장하세요.

In [36]:
## Input Box
ans = 0
X = df.drop(columns=['Outcome','AgeOver30'])
y = df['Outcome']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state = 42, stratify=y)

lr_clf = LogisticRegression(solver='liblinear')
lr_clf.fit (X_train, y_train)
pred = lr_clf.predict(X_test)
pred_proba = lr_clf.predict_proba(X_test)[:, 1]

ans = round(roc_auc_score(y_test, pred_proba), 2)



In [37]:
answer[3] = ans
print(answer[3])

0.81


## Task 4
Task 3에서 학습한 로지스틱 회귀 모델(logis)의 예측 확률(pred_proba)을 활용하여, F1-Score가 최대가 되는 임계값을 소수점 둘째 자리까지 반올림(셋째 자리에서 반올림)하여 구하세요. 해당 값을 answer[4]에 저장하세요.

f1_scores는 아래 공식을 사용합니다.

2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)

Hint : 최대값을 갖는 인덱스를 구하기 위해 np.argmax()를 활용할 수 있습니다.

In [38]:
## Input Box

precisions, recalls, thresholds = precision_recall_curve(y_test, pred_proba)

f1_score = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)

best_idx = np.argmax(f1_score)
ans = round(thresholds[best_idx], 2)


In [39]:
answer[4] = ans
print(answer[4])

0.24


# 정답 확인!

In [40]:
print("=== 작성하신 정답 ===")
for i in range (1, (n+1)) : print("Task " + str(i) + " :: " + str(answer[i]))

=== 작성하신 정답 ===
Task 1 :: 32.46
Task 2 :: 184
Task 3 :: 0.81
Task 4 :: 0.24


In [41]:
### 정답 채점 코드
import hashlib

ANSWER_URL = "https://raw.githubusercontent.com/chaegyeong/KHUDA_ML_10th/f28fb6cf11c7333ac9ba378937d7fe88c62791ba/KHUDA_ML10th_WEEK2/answer.csv"

def md5 (s) : return hashlib.md5(s.encode("utf-8")).hexdigest()

ans_df = pd.read_csv(ANSWER_URL)
gt = {int(r["task"]): str(r["answer"]).strip().lower()
      for _, r in ans_df.iterrows()}

wrong = []

for i in range(1, (n+1)) :
    user_answer = "" if answer[i] is None else md5(str(answer[i]).strip().lower())
    if (user_answer != gt.get(i, "")) : wrong.append(i)

print("탈출!" if not wrong else f"틀린번호 : {wrong}")

탈출!
